# Q3: Association Rule Mining (FP-Growth)

In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Load data and filter price_range == 1
df = pd.read_csv('data/mobile_price.csv')
df_filtered = df[df['price_range'] == 1].copy()
print(f"Samples with price_range=1: {len(df_filtered)}")

# Select the 4 target features
features = ['ram', 'int_memory', 'px_width', 'battery_power']
df_sub = df_filtered[features]
df_sub.describe()

Samples with price_range=1: 500


,ram,int_memory,px_width,battery_power
count,500.000000,500.000000,500.000000,500.000000
mean,1679.490000,32.116000,1251.908000,1228.868000
std,465.850159,18.000739,433.564352,438.614528
min,387.000000,2.000000,500.000000,501.000000
25%,1354.000000,16.000000,878.750000,843.000000
50%,1686.500000,32.000000,1223.000000,1206.000000
75%,2033.750000,47.000000,1629.000000,1596.250000
max,2811.000000,64.000000,1998.000000,1996.000000


## Discretization (3:4:3 value range split) and transaction conversion

In [2]:
def discretize_343(series, name):
    """Discretize a feature into low/medium/high using 3:4:3 value range split."""
    min_val = series.min()
    max_val = series.max()
    range_val = max_val - min_val
    low_upper = min_val + 0.3 * range_val
    med_upper = min_val + 0.7 * range_val
    
    print(f"{name}: min={min_val}, max={max_val}, low<={low_upper:.1f}, medium<={med_upper:.1f}")
    
    result = pd.Series(index=series.index, dtype=str)
    result[series <= low_upper] = f'{name}_low'
    result[(series > low_upper) & (series <= med_upper)] = f'{name}_medium'
    result[series > med_upper] = f'{name}_high'
    return result

# Discretize each feature and build one-hot transaction matrix
transactions = pd.DataFrame(index=df_sub.index)
for feat in features:
    disc = discretize_343(df_sub[feat], feat)
    dummies = pd.get_dummies(disc)
    transactions = pd.concat([transactions, dummies], axis=1)

# Convert to boolean for mlxtend
transactions = transactions.astype(bool)
print(f"\nTransaction matrix shape: {transactions.shape}")
print(f"Columns: {list(transactions.columns)}")
print(f"\nDistribution per category:")
print(transactions.sum())

ram: min=387, max=2811, low<=1114.2, medium<=2083.8
int_memory: min=2, max=64, low<=20.6, medium<=45.4
px_width: min=500, max=1998, low<=949.4, medium<=1548.6
battery_power: min=501, max=1996, low<=949.5, medium<=1547.5

Transaction matrix shape: (500, 12)
Columns: ['ram_high', 'ram_low', 'ram_medium', 'int_memory_high', 'int_memory_low', 'int_memory_medium', 'px_width_high', 'px_width_low', 'px_width_medium', 'battery_power_high', 'battery_power_low', 'battery_power_medium']

Distribution per category:
ram_high                106
ram_low                  53
ram_medium              341
int_memory_high         136
int_memory_low          158
int_memory_medium       206
px_width_high           144
px_width_low            148
px_width_medium         208
battery_power_high      139
battery_power_low       154
battery_power_medium    207
dtype: int64


## 3(a) Frequent patterns with support >= 0.3 (FP-Growth)

In [3]:
# FP-Growth: frequent itemsets with min_support=0.3
freq_items = fpgrowth(transactions, min_support=0.3, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False).reset_index(drop=True)
print(f"Number of frequent patterns (support >= 0.3): {len(freq_items)}\n")
freq_items

Number of frequent patterns (support >= 0.3): 8



,support,itemsets
0,0.682,(ram_medium)
1,0.416,(px_width_medium)
2,0.414,(battery_power_medium)
3,0.412,(int_memory_medium)
4,0.318,"(battery_power_medium, ram_medium)"
5,0.316,(int_memory_low)
6,0.308,(battery_power_low)
7,0.306,"(ram_medium, px_width_medium)"


## 3(b) Association rules: support >= 0.3, confidence >= 0.4, lift >= 0.8

In [4]:
# Generate association rules from frequent itemsets
rules = association_rules(freq_items, metric='support', min_threshold=0.3, num_itemsets=len(transactions))

# Filter: support >= 0.3, confidence >= 0.4, lift >= 0.8
rules_filtered = rules[
    (rules['support'] >= 0.3) &
    (rules['confidence'] >= 0.4) &
    (rules['lift'] >= 0.8)
].sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Number of rules meeting all criteria: {len(rules_filtered)}\n")
rules_filtered[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

Number of rules meeting all criteria: 4



,antecedents,consequents,support,confidence,lift
0,(ram_medium),(battery_power_medium),0.318,0.466276,1.126270
1,(battery_power_medium),(ram_medium),0.318,0.768116,1.126270
2,(ram_medium),(px_width_medium),0.306,0.448680,1.078559
3,(px_width_medium),(ram_medium),0.306,0.735577,1.078559


## 3(c) Observations

**Frequent Patterns (3a):**
- 8 frequent patterns were found with support >= 0.3. The most frequent single item is `ram_medium` (support=0.682), indicating that 68.2% of price_range=1 phones have RAM in the medium range. This is expected since price_range=1 is the second-lowest tier, which aligns with mid-range RAM values.
- The other three medium-level items (`px_width_medium`, `battery_power_medium`, `int_memory_medium`) all have support around 0.41, suggesting that phones in this price tier tend to cluster around the middle range of most hardware specifications.
- `int_memory_low` (0.316) and `battery_power_low` (0.308) also appear as frequent, reflecting that budget-to-mid-range phones also have a notable portion with low-end specs.
- Only 2 two-item patterns pass the support threshold: `{battery_power_medium, ram_medium}` (0.318) and `{px_width_medium, ram_medium}` (0.306), both involving `ram_medium` as the anchor.

**Association Rules (3b):**
- 4 rules pass all three criteria (support >= 0.3, confidence >= 0.4, lift >= 0.8).
- The strongest rules by lift involve `ram_medium` ↔ `battery_power_medium` (lift=1.126) and `ram_medium` ↔ `px_width_medium` (lift=1.079). Lift > 1 indicates positive correlation — medium RAM tends to co-occur with medium battery power and medium pixel width more than expected by chance.
- The directional confidence is asymmetric: `battery_power_medium → ram_medium` has 76.8% confidence, while `ram_medium → battery_power_medium` has only 46.6% confidence. This is because `ram_medium` has a much higher base support (68.2%), so knowing a phone has medium battery power strongly predicts medium RAM, but the reverse is weaker.
- All lift values are close to 1.0 (ranging from 1.08 to 1.13), meaning the associations are statistically positive but not dramatically strong. This suggests that within the price_range=1 group, the four features are somewhat independent, with only mild co-occurrence patterns.